<a href="https://colab.research.google.com/github/JustinMann123/seismic_babes_week8/blob/main/Copy_of_EMSC2010_A2_Weekly_Project_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EMSC2010 Group Project -- A2: Weekly Project 3 -- 	Correlation and regression

## 1. Project Overview
Group name: Seismic Babes

Project week: 9 to 10

Project title: Verifying Omori's Law via Earthquake Aftershock Data

Datasets used (name and source):
- Turkey 2019: https://zenodo.org/records/8089273
- Italy 2016: https://zenodo.org/records/4306165
- Ridgecrest 2019: https://data.mendeley.com/datasets/x8v5wkbj6r/3

## 2. Roles and contributions

| Role | Primary | Deputy | Completed? | Notes |
| :--- | :--- | :--- | :--- | :--- |
| Github & integration | Justin Mann | Edward Obrien| Yes| Add note|
| Data steward | Edward Obrien | Clover Barnard | Yes| Add note|
| Analysis / modelling | Clover Barnard | Justin mann | Yes| Add note|
| Visualisation / interpretation | Frederique Lawrence | Eloise Day | Yes| Add note|
| Narrative | Owen Douglas | Frederique Lawrence | Yes| Add note|
| Quality Control / Reproducibility  | Eloise Day | Owen Douglas | Yes| Add note|



## 4. Pre-submission checklist
* Notebook runs from top to bottom.
* Datafiles are included in the GitHub repository.
* Commits include meaningful information.
* Each group member has included a brief reflection in the notebook.
* Repository has been shared with the teaching team once your project is completed.

# Question

Does Omori's law hold for earthquakes at different fault zones?

# Hypothesis and Justification

Regression modelling of each aftershock data set should fit to Omori's law, however variations due to the quality of the data (ie. background earthquakes) and the magnitude/fault locations may cause disparity.

Earthquakes:
- 2016 Central Italy M6
    - Normal fault (extension)
- 2023 Turkey M7.8
    - Transform/strike-slip fault
- 2019 Ridgecrest California M7.1
    - Strike-slip/intersecting orthogonal faults

# Data Import - Edward

*These files and website links attempt (albeit vaguely) to mimic a standalone workbook which follows FAIR principles, however due to time constraints the completion of a standalone workbook will be addressed later.*

__Files used:__
https://drive.google.com/drive/folders/14tI7zS-gegd2CDPkzftemk6n0lpM9r3P?usp=drive_link


The data used can be found via the following websites (aftershock data from three separate earthquakes):

Turkey 2019
https://zenodo.org/records/8089273
**Complete Automatic Seismic Processor (CASP) procedure**

Italy 2016
https://zenodo.org/records/4306165
**NLL-SSST-coherence earthquake relocations**


Ridgecrest 2019
https://data.mendeley.com/datasets/x8v5wkbj6r/3
**Raw seismic waveform data from 152 Seismic Sensors**


These websites offer catalogues of data in CSV file format which enabled us to extract recordings >90 days after a mainshock event.

All CSV files uploaded contain the entire dataset downloaded from these sites, however both Turkey and Ridgecrest provided data recorded **before a mainshock event** and this was excluded from the dataframe during the coding process.

The data has been successfully reused by other members of the group project and is readable within Google Colab.

__DISCLAIMER: THIS CODE WAS GENERATED BY AI TOOLS CLAUDE AND GEMINI.__

The CSV was adjusted by defining the mainshock date and counting the aftershocks in the following 90 day period. This crux of this code was developed by AI tool Claude and some minor fixes by Gemini within Google Colab. Claude assisted in translating Unix time used for the Ridgecrest data and also in reformatting the spacing of Italy data to remove semicolons used within its CSV.

The results were also plotted breifly within this section to assist with visualising the entire 90 day values.

In [ ]:
import pandas as pd

# Read the CSV
df = pd.read_csv('Turkey_2023.csv')

# Strip any extra spaces from all column names
df.columns = df.columns.str.strip()

# Define the mainshock date based on previous output (adjust if needed)
mainshock_date = pd.to_datetime('2023-02-06 10:24:48')

# Convert date and time components to numeric, coercing errors for robustness
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['month'] = pd.to_numeric(df['month'], errors='coerce')
df['day'] = pd.to_numeric(df['day'], errors='coerce')
df['hour'] = pd.to_numeric(df['hour'], errors='coerce')
df['min'] = pd.to_numeric(df['min'], errors='coerce')
df['sec'] = pd.to_numeric(df['sec'], errors='coerce')

# Create a temporary DataFrame with columns renamed to match pd.to_datetime expectations
temp_date_cols = df[['year', 'month', 'day', 'hour', 'min', 'sec']].rename(columns={
    'min': 'minute',
    'sec': 'second'
})

# Combine the separate date/time columns into one datetime column
df['datetime'] = pd.to_datetime(temp_date_cols)

# Get aftershocks (everything after mainshock)
aftershocks = df[df['datetime'] > mainshock_date].copy()
aftershocks['days_since_mainshock'] = (aftershocks['datetime'] - mainshock_date).dt.total_seconds() / (24 * 3600)

# Filter to first 90 days
aftershocks_90 = aftershocks[aftershocks['days_since_mainshock'] <= 90].copy()

print(f"Mainshock Date: {mainshock_date}")
print(f"\nFound {len(aftershocks_90)} aftershocks in 90 days")
print(f"Date range: {aftershocks_90['datetime'].min()} to {aftershocks_90['datetime'].max()}")

# Count aftershocks per day
turkey_daily_counts = aftershocks_90.groupby(
    aftershocks_90['days_since_mainshock'].astype(int) + 1
).size().reset_index()
turkey_daily_counts.columns = ['days_since_mainshock', 'aftershock_count']
turkey_daily_counts['earthquake'] = 'Turkey_2023'

print("\nFirst 90 days:")
print(turkey_daily_counts.head())
print(f"\nTotal aftershocks in 90 days: {turkey_daily_counts['aftershock_count'].sum()}")

# Set up data for modelling
x_turkey = turkey_daily_counts['days_since_mainshock'].values
y_turkey = turkey_daily_counts['aftershock_count'].values

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.lineplot(y='aftershock_count', x='days_since_mainshock', data=turkey_daily_counts)
plt.title('Daily Aftershock Counts (2023 Turkey Earthquake - First 90 Days)')
plt.ylabel('Aftershock Count')
plt.xlabel('Days Since Mainshock')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Read the CSV with semicolon delimiter
df = pd.read_csv('Italy_2016.csv', sep=';')

# Strip any extra spaces from all column names
df.columns = df.columns.str.strip()

# Define the specific mainshock date (keeping the original hardcoded date for this cell)
mainshock_date = pd.to_datetime('2016-10-30 06:40:18.269')

# Convert date and time components to numeric, coercing errors for robustness
df['YR'] = pd.to_numeric(df['YR'], errors='coerce')
df['MON'] = pd.to_numeric(df['MON'], errors='coerce')
df['DY'] = pd.to_numeric(df['DY'], errors='coerce')
df['HR'] = pd.to_numeric(df['HR'], errors='coerce')
df['MIN'] = pd.to_numeric(df['MIN'], errors='coerce')
df['SEC'] = pd.to_numeric(df['SEC'], errors='coerce')

# Create a temporary DataFrame with columns renamed to match pd.to_datetime expectations
temp_date_cols = df[['YR', 'MON', 'DY', 'HR', 'MIN', 'SEC']].rename(columns={
    'YR': 'year',
    'MON': 'month',
    'DY': 'day',
    'HR': 'hour',
    'MIN': 'minute',
    'SEC': 'second'
})

# Combine the separate date/time columns into one datetime column
df['datetime'] = pd.to_datetime(temp_date_cols)

# Get aftershocks (everything after mainshock)
aftershocks = df[df['datetime'] > mainshock_date].copy()
aftershocks['days_since_mainshock'] = (aftershocks['datetime'] - mainshock_date).dt.total_seconds() / (24 * 3600)

# Filter to first 90 days
aftershocks_90 = aftershocks[aftershocks['days_since_mainshock'] <= 90].copy()

print(f"Mainshock Date: {mainshock_date}")
print(f"\nFound {len(aftershocks_90)} aftershocks in 90 days")
print(f"Date range: {aftershocks_90['datetime'].min()} to {aftershocks_90['datetime'].max()}")

# Count aftershocks per day
italy_daily_counts = aftershocks_90.groupby(
    aftershocks_90['days_since_mainshock'].astype(int) + 1
).size().reset_index()
italy_daily_counts.columns = ['days_since_mainshock', 'aftershock_count']
italy_daily_counts['earthquake'] = 'Italy_2016'

print("\nFirst 90 days:")
print(italy_daily_counts)
print(f"\nTotal aftershocks in 90 days: {italy_daily_counts['aftershock_count'].sum()}")

# Set up data for modelling
x_italy = italy_daily_counts['days_since_mainshock'].values
y_italy = italy_daily_counts['aftershock_count'].values

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.lineplot(y='aftershock_count', x='days_since_mainshock', data=italy_daily_counts)
plt.title('Daily Aftershock Counts (2016 Italy Earthquake - First 90 Days)')
plt.ylabel('Aftershock Count')
plt.xlabel('Days Since Mainshock')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Read the CSV
df = pd.read_csv('Ridgecrest_2019.csv')

# Convert Unix timestamp to datetime
mainshock_date = pd.to_datetime(1562383080, unit='s')

# Convert time column to datetime
df['datetime'] = pd.to_datetime(df['time'], unit='s')

# Get aftershocks (everything after mainshock)
aftershocks = df[df['datetime'] > mainshock_date].copy()
aftershocks['days_since_mainshock'] = (aftershocks['datetime'] - mainshock_date).dt.total_seconds() / (24 * 3600)

# Filter to first 90 days
aftershocks_90 = aftershocks[aftershocks['days_since_mainshock'] <= 90].copy()

print(f"Mainshock Date (Unix timestamp {1559349029}): {mainshock_date}")
print(f"\nFound {len(aftershocks_90)} aftershocks in 90 days")
print(f"Date range: {aftershocks_90['datetime'].min()} to {aftershocks_90['datetime'].max()}")

# Count aftershocks per day
california_daily_counts = aftershocks_90.groupby(
    aftershocks_90['days_since_mainshock'].astype(int) + 1
).size().reset_index()
california_daily_counts.columns = ['days_since_mainshock', 'aftershock_count']
california_daily_counts['earthquake'] = 'Ridgecrest_2019'

print("\nFirst 90 days:")
print(california_daily_counts)
print(f"\nTotal aftershocks in 90 days: {california_daily_counts['aftershock_count'].sum()}")

# Set up data for modelling
x_california = california_daily_counts['days_since_mainshock'].values
y_california = california_daily_counts['aftershock_count'].values


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.lineplot(y='aftershock_count', x='days_since_mainshock', data=california_daily_counts)
plt.title('Daily Aftershock Counts (2019 Ridgecrest Earthquake - First 90 Days)')
plt.ylabel('Aftershock Count')
plt.xlabel('Days Since Mainshock')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Analysis / Modelling - Clover

In [ ]:
!pip install bambi #system command to install bambi package

In [ ]:
import numpy as np
import bambi as bmb
import arviz as az
import pandas as pd
import pymc as pm

Plot the the X and Y data for each earthquake to visualise correlation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].plot(x_california,y_california,'ok')
axes[0].set_title('2019 Ridgecrest Earthquake')
axes[0].set_xlabel('Days since mainshock')
axes[0].set_ylabel('Number of aftershocks')
axes[0].minorticks_on()

axes[1].plot(x_italy,y_italy,'ok')
axes[1].set_title('2016 Italy Earthquake')
axes[1].set_xlabel('Days since mainshock')
axes[1].set_ylabel('Number of aftershocks')
axes[1].minorticks_on()

axes[2].plot(x_turkey,y_turkey,'ok')
axes[2].set_title('2023 Turkey Earthquake')
axes[2].set_xlabel('Days since mainshock')
axes[2].set_ylabel('Number of aftershocks')
axes[2].minorticks_on()

plt.tight_layout()

#### Ridgecrest, California 2019 Regression Modelling via ```bambi``` package

In [ ]:
# Log-transform x and y for power law fitting
log_x_california = np.log(x_california)
log_y_california = np.log(y_california)

# Build a dataframe for Bambi
california_log_df = pd.DataFrame({'log_days': log_x_california, 'log_count': log_y_california})

# Fit using Bambi
model_california = bmb.Model('log_count ~ log_days', california_log_df)
idata_california = model_california.fit(draws=2000, tune=1000, chains=6,progressbar=False)

# Predict across days 1-90 in log space
x_range = np.linspace(1, 90, 100)
log_x_range = np.log(x_range)

# Draw from posterior
alpha_draws = idata_california.posterior["Intercept"].values.flatten()
beta_draws  = idata_california.posterior["log_days"].values.flatten()

# Compute posterior draws of log(y), then back-transform
log_y_draws = alpha_draws[:, None] + beta_draws[:, None] * log_x_range[None, :]
y_draws = np.exp(log_y_draws)  # back to original scale

# Posterior mean and HDI
posterior_mean_california = y_draws.mean(axis=0)
hdi_california = az.hdi(y_draws, hdi_prob=0.95)

# Plot on log-log scale
plt.figure(figsize=(10, 6))
plt.plot(log_x_california, log_y_california, 'ok', label="Data")
plt.plot(log_x_range, np.log(posterior_mean_california), color="C1", label="Posterior mean")
plt.fill_between(
    log_x_range,
    np.log(hdi_california[:, 0]),
    np.log(hdi_california[:, 1]),
    alpha=0.3,
    color="C1",
    label="95% HDI",
    edgecolor=None
)
plt.xlabel('Log(Days since mainshock)')
plt.ylabel('Log(Aftershock count)')
plt.title('Ridgecrest 2019 — Log-Log Scale')
plt.legend()
plt.minorticks_on()

# Plot against Power
plt.figure(figsize=(10, 6))
plt.plot(x_california, y_california, 'ok', label="Data")
plt.plot(x_range, posterior_mean_california, color="C1", label="Posterior mean")
plt.fill_between(
    x_range,
    hdi_california[:, 0],
    hdi_california[:, 1],
    alpha=0.3,
    color="C1",
    label="95% HDI",
    edgecolor=None
)
plt.xlabel('Days since mainshock')
plt.ylabel('Aftershock count')
plt.title('Ridgecrest 2019 — Power Law fit')
plt.legend()
plt.minorticks_on()

#### Central Italy 2016 Regression Modelling via ```bambi``` package

In [ ]:
# Log-transform x and y for power law fitting
log_x_italy = np.log(x_italy)
log_y_italy = np.log(y_italy)

# Build a dataframe for Bambi
italy_log_df = pd.DataFrame({'log_days': log_x_italy, 'log_count': log_y_italy})

# Fit using Bambi
model_italy = bmb.Model('log_count ~ log_days', italy_log_df)
idata_italy = model_italy.fit(draws=2000, tune=1000, chains=6,progressbar=False)

# Predict across days 1-90 in log space
x_range = np.linspace(1, 90, 100)
log_x_range = np.log(x_range)

# Draw from posterior
alpha_draws = idata_italy.posterior["Intercept"].values.flatten()
beta_draws  = idata_italy.posterior["log_days"].values.flatten()

# Compute posterior draws of log(y), then back-transform
log_y_draws = alpha_draws[:, None] + beta_draws[:, None] * log_x_range[None, :]
y_draws = np.exp(log_y_draws)  # back to original scale

# Posterior mean and HDI
posterior_mean_italy = y_draws.mean(axis=0)
hdi_italy = az.hdi(y_draws, hdi_prob=0.95)

# Plot on log-log scale
plt.figure(figsize=(10, 6))
plt.plot(log_x_italy, log_y_italy, 'ok', label="Data")
plt.plot(log_x_range, np.log(posterior_mean_italy), color="C1", label="Posterior mean")
plt.fill_between(
    log_x_range,
    np.log(hdi_italy[:, 0]),
    np.log(hdi_italy[:, 1]),
    alpha=0.3,
    color="C1",
    label="95% HDI",
    edgecolor=None
)
plt.xlabel('Log(Days since mainshock)')
plt.ylabel('Log(Aftershock count)')
plt.title('Italy 2016 — Log-Log Scale')
plt.legend()
plt.minorticks_on()

# Plot against Power
plt.figure(figsize=(10, 6))
plt.plot(x_italy, y_italy, 'ok', label="Data")
plt.plot(x_range, posterior_mean_italy, color="C1", label="Posterior mean")
plt.fill_between(
    x_range,
    hdi_italy[:, 0],
    hdi_italy[:, 1],
    alpha=0.3,
    color="C1",
    label="95% HDI",
    edgecolor=None
)
plt.xlabel('Days since mainshock')
plt.ylabel('Aftershock count')
plt.title('Italy 2016 — Power Law fit')
plt.legend()
plt.minorticks_on()

#### Turkey 2023 Regression Modelling via ```bambi``` package**

In [ ]:
# Log-transform x and y for power law fitting
log_x_turkey = np.log(x_turkey)
log_y_turkey = np.log(y_turkey)

# Build a dataframe for Bambi
turkey_log_df = pd.DataFrame({'log_days': log_x_turkey, 'log_count': log_y_turkey})

# Fit using Bambi
model_turkey = bmb.Model('log_count ~ log_days', turkey_log_df)
idata_turkey = model_turkey.fit(draws=2000, tune=1000, chains=6,progressbar=False)

# Predict across days 1-90 in log space
x_range = np.linspace(1, 90, 100)
log_x_range = np.log(x_range)

# Draw from posterior
alpha_draws = idata_turkey.posterior["Intercept"].values.flatten()
beta_draws  = idata_turkey.posterior["log_days"].values.flatten()

# Compute posterior draws of log(y), then back-transform
log_y_draws = alpha_draws[:, None] + beta_draws[:, None] * log_x_range[None, :]
y_draws = np.exp(log_y_draws)  # back to original scale

# Posterior mean and HDI
posterior_mean_turkey = y_draws.mean(axis=0)
hdi_turkey = az.hdi(y_draws, hdi_prob=0.95)

# Plot on log-log scale
plt.figure(figsize=(10, 6))
plt.plot(log_x_turkey, log_y_turkey, 'ok', label="Data")
plt.plot(log_x_range, np.log(posterior_mean_turkey), color="C1", label="Posterior mean")
plt.fill_between(
    log_x_range,
    np.log(hdi_turkey[:, 0]),
    np.log(hdi_turkey[:, 1]),
    alpha=0.3,
    color="C1",
    label="95% HDI",
    edgecolor=None
)
plt.xlabel('Log(Days since mainshock)')
plt.ylabel('Log(Aftershock count)')
plt.title('Turkey 2023 — Log-Log Scale')
plt.legend()
plt.minorticks_on()

# Plot against Power
plt.figure(figsize=(10, 6))
plt.plot(x_turkey, y_turkey, 'ok', label="Data")
plt.plot(x_range, posterior_mean_turkey, color="C1", label="Posterior mean")
plt.fill_between(
    x_range,
    hdi_turkey[:, 0],
    hdi_turkey[:, 1],
    alpha=0.3,
    color="C1",
    label="95% HDI",
    edgecolor=None
)
plt.xlabel('Days since mainshock')
plt.ylabel('Aftershock count')
plt.title('Turkey 2023 — Power Law fit')
plt.legend()
plt.minorticks_on()

# Visualisation - Freddie

### Goals
1.   Plot all 3 graphs, remove the points and format better
2.   Average our graphs together, to create one average graph between the 3
1.   Plot Omori's law onto each individual graph and onto the average Graph
2.   Extract an approximate equation from the average graph and compare to Omori's Law











In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Adapted from AI
# Clean + align data
def prepare_data(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)

    # Ensure same length
    n = min(len(x), len(y))
    x = x[:n]
    y = y[:n]

    # Remove NaNs only (NOT zeros)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    return x, y

# Confidence band
def confidence_band(x, y, y_fit, confidence=0.90):
    n = len(x)
    residuals = y - y_fit
    s_err = np.sqrt(np.sum(residuals**2) / (n - 2))

    x_mean = np.mean(x)
    t_val = 1.645  # ~90% confidence

    conf = t_val * s_err * np.sqrt(
        1/n + (x - x_mean)**2 / np.sum((x - x_mean)**2)
    )

    return conf

# Figure
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

datasets = [
    ("2019 Ridgecrest Earthquake", x_california, y_california),
    ("2016 Italy Earthquake", x_italy, y_italy),
    ("2023 Turkey Earthquake", x_turkey, y_turkey)
]

for i, (title, x_raw, y_raw) in enumerate(datasets):

    x, y = prepare_data(x_raw, y_raw)

    if len(x) < 2:
        print(f"{title} skipped — no usable data")
        continue

    # Shift x to avoid log(0), but keep ALL points
    x_shift = x + 1

    #Log regression
    ax = axes[0, i]
    ax.plot(x, y, 'ok')

    log_x = np.log(x_shift)
    coef = np.polyfit(log_x, y, 1)

    x_fit = np.linspace(min(x), max(x), 200)
    y_fit = coef[0] * np.log(x_fit + 1) + coef[1]

    ax.plot(x_fit, y_fit, 'r-')

    # Confidence band
    y_pred = coef[0] * log_x + coef[1]
    conf = confidence_band(log_x, y, y_pred)

    y_upper = y_fit + np.mean(conf)
    y_lower = y_fit - np.mean(conf)

    ax.fill_between(x_fit, y_lower, y_upper, alpha=0.2)

    ax.set_title(f"{title} (Log Fit)")
    ax.set_xlabel("Days since mainshock")
    ax.set_ylabel("Aftershocks")
    ax.minorticks_on()

    #Power law (log-log fit)
    ax = axes[1, i]
    ax.plot(x, y, 'ok')

    # Only remove y <= 0 for log(y)
    mask = y > 0
    x_p = x_shift[mask]
    y_p = y[mask]

    if len(x_p) < 2:
        print(f"{title} power law skipped — insufficient positive data")
        continue

    coef = np.polyfit(np.log(x_p), np.log(y_p), 1)
    a = np.exp(coef[1])
    b = -coef[0]

    x_fit = np.linspace(min(x), max(x), 200)
    y_fit = a * (x_fit + 1)**(-b)

    ax.plot(x_fit, y_fit, 'r-')

    # Confidence band (in log space)
    y_log_pred = coef[0] * np.log(x_p) + coef[1]
    conf = confidence_band(np.log(x_p), np.log(y_p), y_log_pred)

    y_upper = a * (x_fit + 1)**(-(b - np.mean(conf)))
    y_lower = a * (x_fit + 1)**(-(b + np.mean(conf)))

    ax.fill_between(x_fit, y_lower, y_upper, alpha=0.2)

    ax.set_title(f"{title} (Power Law Fit)")
    ax.set_xlabel("Days since mainshock")
    ax.set_ylabel("Aftershocks")
    ax.minorticks_on()

plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Adapted from AI
#Prepare data
def prepare_data(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)

    n = min(len(x), len(y))
    x = x[:n]
    y = y[:n]

    mask = np.isfinite(x) & np.isfinite(y)
    return x[mask], y[mask]

#Models
def log_model_fit(x, y, x_fit):
    x_shift = x + 1
    coef = np.polyfit(np.log(x_shift), y, 1)
    return coef[0] * np.log(x_fit + 1) + coef[1]

def power_model_fit(x, y, x_fit):
    x_shift = x + 1
    mask = y > 0
    x_p = x_shift[mask]
    y_p = y[mask]

    coef = np.polyfit(np.log(x_p), np.log(y_p), 1)
    a = np.exp(coef[1])
    b = -coef[0]

    return a * (x_fit + 1)**(-b)

#Bootstrap HDI
def bootstrap_hdi(x, y, model_func, x_fit, n_boot=1000, alpha=0.05):
    y_boot = []
    n = len(x)

    for _ in range(n_boot):
        idx = np.random.randint(0, n, n)
        x_s = x[idx]
        y_s = y[idx]

        try:
            y_fit = model_func(x_s, y_s, x_fit)
            y_boot.append(y_fit)
        except:
            continue

    y_boot = np.array(y_boot)

    lower = np.percentile(y_boot, 100 * (alpha/2), axis=0)
    upper = np.percentile(y_boot, 100 * (1 - alpha/2), axis=0)

    return lower, upper

#Figure 1: 4 subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

datasets = [
    ("California (2019 Ridgecrest)", x_california, y_california),
    ("Turkey (2023)", x_turkey, y_turkey)
]

stored_results = []  # store for combined plot

for i, (title, x_raw, y_raw) in enumerate(datasets):

    x, y = prepare_data(x_raw, y_raw)
    if len(x) < 2:
        continue

    x_fit = np.linspace(min(x), max(x), 200)

    # Log
    ax = axes[0, i]

    y_fit_log = log_model_fit(x, y, x_fit)
    lower_log, upper_log = bootstrap_hdi(x, y, log_model_fit, x_fit)

    ax.plot(x_fit, y_fit_log, 'r-', label="Log model")
    ax.fill_between(x_fit, lower_log, upper_log, color='yellow', alpha=0.3, label="95% HDI")

    ax.set_title(f"{title} (Log)")
    ax.set_xlabel("Days since mainshock")
    ax.set_ylabel("Aftershocks")
    ax.legend()
    ax.minorticks_on()

    # Power
    ax = axes[1, i]

    y_fit_pow = power_model_fit(x, y, x_fit)
    lower_pow, upper_pow = bootstrap_hdi(x, y, power_model_fit, x_fit)

    ax.plot(x_fit, y_fit_pow, 'b-', label="Power law")
    ax.fill_between(x_fit, lower_pow, upper_pow, color='yellow', alpha=0.3, label="95% HDI")

    ax.set_title(f"{title} (Power Law)")
    ax.set_xlabel("Days since mainshock")
    ax.set_ylabel("Aftershocks")
    ax.legend()
    ax.minorticks_on()

    # store for combined plot
    stored_results.append((title, x_fit, y_fit_log, y_fit_pow))

plt.tight_layout()
plt.show()

# Figure 2: Combined comparison
plt.figure(figsize=(8, 6))

for title, x_fit, y_log, y_pow in stored_results:
    plt.plot(x_fit, y_log, '--', label=f"{title} (Log)")
    plt.plot(x_fit, y_pow, '-', label=f"{title} (Power)")

plt.xlabel("Days since mainshock")
plt.ylabel("Aftershocks")
plt.title("Model Comparison Across Earthquakes")
plt.legend()
plt.minorticks_on()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#Prepare data
def prepare_data(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)

    n = min(len(x), len(y))
    x = x[:n]
    y = y[:n]

    mask = np.isfinite(x) & np.isfinite(y) & (y > 0)
    return x[mask], y[mask]

# Log model
def log_model_fit(x, y, x_fit):
    x_shift = x + 1
    coef = np.polyfit(np.log(x_shift), y, 1)
    return coef[0] * np.log(x_fit + 1) + coef[1]

#Bootstrap HDI
def bootstrap_hdi(x, y, x_fit, n_boot=800, alpha=0.05):
    y_boot = []
    n = len(x)

    for _ in range(n_boot):
        idx = np.random.randint(0, n, n)
        x_s = x[idx]
        y_s = y[idx]

        try:
            x_shift = x_s + 1
            coef = np.polyfit(np.log(x_shift), y_s, 1)
            y_fit = coef[0] * np.log(x_fit + 1) + coef[1]
            y_boot.append(y_fit)
        except:
            continue

    y_boot = np.array(y_boot)

    lower = np.percentile(y_boot, 2.5, axis=0)
    upper = np.percentile(y_boot, 97.5, axis=0)

    return lower, upper

# Data
datasets = [
    ("California", x_california, y_california),
    ("Italy", x_italy, y_italy),
    ("Turkey", x_turkey, y_turkey)
]

#Clean and store
cleaned = []
for name, x, y in datasets:
    x, y = prepare_data(x, y)
    cleaned.append((name, x, y))

#Combine the datasets
x_all = np.concatenate([d[1] for d in cleaned])
y_all = np.concatenate([d[2] for d in cleaned])

x_all, y_all = prepare_data(x_all, y_all)

x_fit = np.linspace(min(x_all), max(x_all), 300)

# Average model
y_avg = log_model_fit(x_all, y_all, x_fit)
low_avg, up_avg = bootstrap_hdi(x_all, y_all, x_fit)

# Figure 1: Average Only
plt.figure(figsize=(8, 6))

plt.plot(x_fit, y_avg, 'r-', label="Average decay model (3 earthquakes)")
plt.fill_between(x_fit, low_avg, up_avg, color='yellow', alpha=0.3, label="95% HDI")

plt.title("Average Aftershock Decay Curve\n(California + Italy + Turkey)")
plt.xlabel("Days since mainshock")
plt.ylabel("Aftershocks")
plt.legend()
plt.minorticks_on()
plt.tight_layout()
plt.show()

# Figure 2: individual and average

plt.figure(figsize=(9, 6))

colors = ["blue", "green", "orange"]

for i, (name, x, y) in enumerate(cleaned):

    x_fit_i = np.linspace(min(x), max(x), 200)
    y_fit_i = log_model_fit(x, y, x_fit_i)

    plt.plot(x_fit_i, y_fit_i, color=colors[i], linestyle='--', label=f"{name}")

# average overlay
plt.plot(x_fit, y_avg, 'r-', linewidth=2.5, label="Average model (all 3)")
plt.fill_between(x_fit, low_avg, up_avg, color='yellow', alpha=0.2, label="95% HDI (average)")

plt.title("Comparison of Aftershock Decay Models\nvs Global Average")
plt.xlabel("Days since mainshock")
plt.ylabel("Aftershocks")

plt.legend()
plt.minorticks_on()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Omori's Law
def omori(t, k, c, p):
    return k / (t + c)**p

#Time axis
t = np.linspace(0, 30, 500)

#Basic Parameters
c = 0.5
p = 1.1

#Magnitude-specific k values
k_values = {
    "Mw 6.0": 600,
    "Mw 7.1": 1000,
    "Mw 7.8": 2000
}

# Mw 6.0 Graph

plt.figure(figsize=(7, 5))
plt.plot(t, omori(t, k_values["Mw 6.0"], c, p), 'r-', label="Omori’s Law")

plt.title("Omori’s Law Aftershock Decay (Mw 6.0)")
plt.xlabel("Days since mainshock")
plt.ylabel("Aftershock rate n(t)")
plt.legend()
plt.minorticks_on()
plt.tight_layout()
plt.show()

# Mw 7.1 Graph
plt.figure(figsize=(7, 5))
plt.plot(t, omori(t, k_values["Mw 7.1"], c, p), 'b-', label="Omori’s Law")

plt.title("Omori’s Law Aftershock Decay (Mw 7.1)")
plt.xlabel("Days since mainshock")
plt.ylabel("Aftershock rate n(t)")
plt.legend()
plt.minorticks_on()
plt.tight_layout()
plt.show()

# Mw 7.8 Graph
plt.figure(figsize=(7, 5))
plt.plot(t, omori(t, k_values["Mw 7.8"], c, p), 'g-', label="Omori’s Law")

plt.title("Omori’s Law Aftershock Decay (Mw 7.8)")
plt.xlabel("Days since mainshock")
plt.ylabel("Aftershock rate n(t)")
plt.legend()
plt.minorticks_on()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Shared functions
def prepare_data(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)

    n = min(len(x), len(y))
    x, y = x[:n], y[:n]

    mask = np.isfinite(x) & np.isfinite(y) & (y > 0)
    return x[mask], y[mask]

def omori(t, k, c, p):
    return k / (t + c)**p

def power_model_fit(x, y, x_fit):
    coef = np.polyfit(np.log(x + 1), np.log(y), 1)
    a = np.exp(coef[1])
    b = -coef[0]
    return a * (x_fit + 1)**(-b)

def fit_omori(x, y, x_fit):
    popt, _ = curve_fit(
        omori,
        x + 1,
        y,
        p0=[max(y), 0.5, 1.0],
        bounds=([0, 0.01, 0.5], [np.inf, 10, 2.5]),
        maxfev=20000
    )
    return omori(x_fit + 1, *popt)

# Dataset wrapper
datasets = [
    ("Italy 2016", x_italy, y_italy),
    ("Turkey 2023 (Mw 7.8)", x_turkey, y_turkey),
    ("Ridgecrest 2019 (Mw 7.1)", x_california, y_california)
]

#Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (name, x_raw, y_raw) in enumerate(datasets):

    x, y = prepare_data(x_raw, y_raw)
    idx = np.argsort(x)
    x, y = x[idx], y[idx]

    x_fit = np.linspace(0, max(x), 400)

    # models
    y_power = power_model_fit(x, y, x_fit)
    y_omori = fit_omori(x, y, x_fit)

    ax = axes[i]

    ax.plot(x_fit, y_power, 'b-', label="Power law")
    ax.plot(x_fit, y_omori, 'r-', label="Omori’s Law")

    ax.set_title(name)
    ax.set_xlabel("Days")
    ax.set_ylabel("Aftershock rate")
    ax.legend()
    ax.minorticks_on()

plt.tight_layout()
plt.show()

# Interpretation - Owen
## Model Interpretation

- Power laws predicted far higher initial earthquakes than presented by the data with more data points falling outside the uncertainty interval, whereas log laws saw a larger amount of data points within the interval, as well as a significantly improved representation of the average. This is reflected in a divergence between models from day 0, where they eventually agree as time scales are increased.
- The initial discrepancy with Omori’s Law is a number of data points falling outside the trend, aligning more similarly with the log trend. This is mostly due to the time period, where there are a number of earthquakes possibly caused by redistributions along the fault line and secondary triggers from the larger aftershock earthquakes. This falls within expected post-rupture processes - with a number of residual earthquakes contradicting Omori’s law and convoluting the data.
- Uncertainty intervals are largest at the beginning and end of the time periods, reflecting unreliability in data at these points. This can be confirmed with the statements above and in the limitations section, where a number of influential factors disperse the data and expose a difference between the models. This can also be confirmed through Omori’s Law, where significant anomalies in data have disproportionate influence over its trend due to its log-log nature.


## Physical Interpretation
- Log laws assume the relationship between earthquake amount and frequency is magnitude dependent and will never reach 0 (implying that there will always be residual earthquakes because of the initial event), whereas the power law is justified by Omori’s Law (as it reaches 0 eventually) and is practical - this is also justified by the data as it reaches ~80 days range.
- Beyond day 20, the data settles into a classical stress-relaxation decay, falling into Omori’s Law’s description rather than the log representations.
- Turkey’s lower aftershock rate (than other earthquakes) despite higher magnitude can be attributed to its rupture location and multi-segment strike-slip action. This would have released stress more evenly along the fault and thus prevented a number of aftershocks, especially when compared to Ridgecrest (of which was a result of a shear zone).


## Limitations
- We also saw a number of significant data anomalies within the models that would have potentially influenced our trends. Most notably, Ridgecrest had a disproportionate number of recorded aftershocks initially (a few days after the event), and Italy experienced a disproportionate amount of aftershocks >80 days after the initial event. These would have had influence over the overall trend of both scenarios.
- The scenarios also occurred in different ridge geometries, with locations like Turkey occurring in a strike slip fault, Ridgecrest occurring in a shear zone, and Italy’s multi-faceted fault zone. These may have also influenced the amount of residual earthquakes, as they differ in activity, and further, may have heightened the magnitude of said residual quakes, and thus caused more than would have occurred in other scenarios.


# Individual Reflections

### Data Steward - Ed

Gathering clean and concise aftershock data of three earthquakes was quite difficult. The data was often difficult to narrow down and interpret. I spent most of my time processing the data from CSV files downloaded because the mainshock events I was looking for (as a startpoint) were slightly buried within some of the files. Clover was a great help as per usual; I was able to discuss problems and she would offer plenty of resources, including the sources that were used for the data.

A problem from the beginning was deciding on appropriate earthquake events. Older earthquakes such as Landers (1992) and Hector Mine (1999) were difficult to assess due to the availability and reliability of data recorded. Initially, I took to navigating the USGS website to download CSV files of aftershock data, however this website did not quite offer the information I was looking for as I struggled to acquire enough aftershock events to make any reasonable conclusions.

This problem puzzled me because I was sifting through a lot of detailed measurements whilst struggling to find something so seemingly basic as an amount of aftershocks. I also struggled to manually interpret data because of the sheer size of some of the csv files and therefore I did use AI tools to assist me.

I am responsible for the imperfect presentation of data + metadata which is not organised within a standalone source (outside of CSV files being grouped into a google drive folder) and the presentation/adherence to FAIR principles is not thorough. Due to the difficulty of efficiently collecting data this problem will need to be addressed later.

### Analysis and Modelling - Clover

This week presented a significant challenge finding appropriate data for our analysis. Originally we intended to analyse three earthquakes of the same magnitude and three additional earthquakes of a different magnitude to compare how Omori’s law might vary with magnitude. However we had difficulty finding appropriate datasets that were limited to only the aftershock recording so switched tactics to focus on three events we could find specific datasets for. I helped Ed find some these better options and once he’d loaded and cleaned the data sets I applied the Bambi regression modelling we did in class. Importantly in order to compare our result to Omori’s law I had to fit the data to a power law, this was done by log transforming the data. After initially plotting the data as a power law fit I realised the best way to assess it’s the fit was by using a log-log scale. Hence I produced both plots, allowing Freddie to further expand on the visualisation and providing Owen with data to assess for the interpretation.

There was also an issue this week of two of us editing the document at the same time (remotely), it caused some of my code to not save however we were able to communicate and ensure only one person was editing at the same time and this is something we will do going forward.

### Visualisation and Interpretation - Freddie

As the visualisation lead, my role was to take the amazing graphs made by CLover and re-format them so that they could be interpreted relative to Omori's calculated law. This took quite a while as it was very difficult to firsly model Omori's law and then try and get it on the same graphs as the Earthquakes we modeled. I had a lot of difficulty trying to gfet the Italy graph to work, it was very tempremental and would often break. Overall, this week was the hardest one yet coding-wise, but I really enjoyed the actual concept of the project, as I think Omori's law is very interesting.

### Narrative - Owen
I found my role this week as the narrative lead moderately challenging, as I was tasked with first making sure the code followed a logical path and reached its desired outcome, and second assisting Freddie with interpreting the concluding data. The clear steps taken by my repsective group members made narrating the code fairly easy, simply explaining each step taken with a header above each set of code. My main challenge was to interpret the data with Freddie - of which ended up being fairly straightforward (after my wifi began working again and my work would actually save to the document). Overall, I found my role this week reasonably challenging, and I owe a lot of it to my teammates who gave me logical and clear code!

### Github and Integration - Justin

My role this week was relatively straightforward. It allowed me to appreciate the importance of a collaborative team. Good teamwork and communication helped ensure every part of this process was completed on time. My primary responsibility was integration. The real challenge was merging everyone’s individual analysis scripts and components without overwriting progress. In previous weeks, the team grew complacent in handling the script and data files. This week, we focused on avoiding that. We addressed the issue by running the final commit and files in the GitHub repository. This prevented the problem we faced last week. Hopefully! Another positive takeaway was communicating actively and setting deadlines. This proved effective for managing expectations and allowed for a smooth project week.

### Quality Control and Reproducibility - Ellie




I had a lot of assignments this week so chose quality control as my role, because this job is relatively minor. The main things I did were looking through the code, making sure it all ran correctly and the naming was consistent, especially as it can be hard to code in the exact same way as your group members. I checked in my group members and the code every couple of days and it always looked good. I also made sure the code ran properly, which it did. My group members did an amazing job this week and I barely needed to change anything because of it. Overall, this week has been quite easy for compared to the previous weeks as I did not have as much to do or as big of a role as previously. I feel as though I completed my role to a good standard.